# Task 3 — Batch CycleGAN Checkpoint Evaluation

This notebook evaluates every saved generator checkpoint in a selected epoch range.

It:

- discovers checkpoints automatically;
- generates 300 images in both directions for each checkpoint;
- follows the supplied FID and MiFID evaluation method;
- reuses real-image Inception features to reduce repeated work;
- saves separate outputs for every checkpoint;
- displays and saves one final comparison table.

The default configuration evaluates epochs **60 through 100**, in steps of 5, from `task3_gan_diffaug_ema`.

## 1. Install and import packages

In [1]:
%pip install -q scipy pillow tqdm pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import scipy.linalg
from scipy.spatial.distance import cosine

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms.functional import to_pil_image


SEED = 5775
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 5090


## 2. Configure checkpoint range and paths

In [3]:
# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

DATA_PROJECT_ROOT = Path("/app/task3_gan")

EXPERIMENT_ROOT = Path(
    "/app/task3_gan_diffaug_ema"
)

CHECKPOINT_DIR = EXPERIMENT_ROOT / "checkpoints"

REAL_MONET = (
    DATA_PROJECT_ROOT / "data" / "monet_jpg"
)

REAL_PHOTO = (
    DATA_PROJECT_ROOT / "data" / "photo_jpg"
)

OUTPUT_ROOT = (
    EXPERIMENT_ROOT
    / "batch_official_evaluation_300"
)


# ------------------------------------------------------------
# Checkpoint range
# ------------------------------------------------------------

START_EPOCH = 60
END_EPOCH = 100
EPOCH_STEP = 5

# Also evaluate best_cycle_checkpoint.pt after epoch snapshots.
INCLUDE_BEST_CYCLE = True


# ------------------------------------------------------------
# Evaluation settings
# ------------------------------------------------------------

N_EVAL = 300
GENERATION_BATCH_SIZE = 16
METRIC_BATCH_SIZE = 32
NUM_WORKERS = 0

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
}


def list_images(folder):
    return sorted(
        path
        for path in folder.iterdir()
        if (
            path.is_file()
            and path.suffix.lower() in IMAGE_EXTENSIONS
        )
    )


for folder in [REAL_MONET, REAL_PHOTO, CHECKPOINT_DIR]:
    if not folder.is_dir():
        raise FileNotFoundError(
            f"Required folder not found: {folder}"
        )

real_monet_paths = list_images(REAL_MONET)
real_photo_paths = list_images(REAL_PHOTO)

if (
    len(real_monet_paths) < N_EVAL
    or len(real_photo_paths) < N_EVAL
):
    raise ValueError(
        f"At least {N_EVAL} images are required "
        "in both real-image folders."
    )

real_monet_paths = real_monet_paths[:N_EVAL]
real_photo_paths = real_photo_paths[:N_EVAL]

checkpoint_paths = []
missing_checkpoints = []

for epoch in range(
    START_EPOCH,
    END_EPOCH + 1,
    EPOCH_STEP,
):
    checkpoint_path = (
        CHECKPOINT_DIR
        / f"generators_epoch_{epoch:03d}.pt"
    )

    if checkpoint_path.is_file():
        checkpoint_paths.append(checkpoint_path)
    else:
        missing_checkpoints.append(checkpoint_path.name)

if INCLUDE_BEST_CYCLE:
    best_cycle_path = (
        CHECKPOINT_DIR / "best_cycle_checkpoint.pt"
    )

    if best_cycle_path.is_file():
        checkpoint_paths.append(best_cycle_path)
    else:
        missing_checkpoints.append(best_cycle_path.name)

if not checkpoint_paths:
    raise FileNotFoundError(
        "No checkpoints were found in the selected range."
    )

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Experiment:", EXPERIMENT_ROOT)
print("Real Monet images:", len(real_monet_paths))
print("Real photo images:", len(real_photo_paths))
print("Checkpoints found:", len(checkpoint_paths))

for checkpoint_path in checkpoint_paths:
    print(" -", checkpoint_path.name)

if missing_checkpoints:
    print("Missing checkpoints skipped:")

    for checkpoint_name in missing_checkpoints:
        print(" -", checkpoint_name)

Experiment: /app/task3_gan_diffaug_ema
Real Monet images: 300
Real photo images: 300
Checkpoints found: 10
 - generators_epoch_060.pt
 - generators_epoch_065.pt
 - generators_epoch_070.pt
 - generators_epoch_075.pt
 - generators_epoch_080.pt
 - generators_epoch_085.pt
 - generators_epoch_090.pt
 - generators_epoch_095.pt
 - generators_epoch_100.pt
 - best_cycle_checkpoint.pt


## 3. Recreate the trained generator

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, kernel_size=3),
            nn.InstanceNorm2d(channels),
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(
        self,
        input_channels=3,
        output_channels=3,
        num_residual_blocks=9,
    ):
        super().__init__()

        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(
                input_channels,
                64,
                kernel_size=7,
            ),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
        ]

        channels = 64

        for _ in range(2):
            layers.extend([
                nn.Conv2d(
                    channels,
                    channels * 2,
                    kernel_size=4,
                    stride=2,
                    padding=1,
                ),
                nn.InstanceNorm2d(channels * 2),
                nn.ReLU(inplace=True),
            ])

            channels *= 2

        for _ in range(num_residual_blocks):
            layers.append(ResidualBlock(channels))

        for _ in range(2):
            layers.extend([
                nn.ConvTranspose2d(
                    channels,
                    channels // 2,
                    kernel_size=4,
                    stride=2,
                    padding=1,
                ),
                nn.InstanceNorm2d(channels // 2),
                nn.ReLU(inplace=True),
            ])

            channels //= 2

        layers.extend([
            nn.ReflectionPad2d(3),
            nn.Conv2d(
                channels,
                output_channels,
                kernel_size=7,
            ),
            nn.Tanh(),
        ])

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


G_AB = Generator().to(device)  # Photo -> Monet
G_BA = Generator().to(device)  # Monet -> photo

G_AB.eval()
G_BA.eval()


def load_generator_checkpoint(checkpoint_path):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    G_AB.load_state_dict(
        checkpoint["G_AB_state_dict"]
    )

    G_BA.load_state_dict(
        checkpoint["G_BA_state_dict"]
    )

    G_AB.eval()
    G_BA.eval()

    return checkpoint.get("epoch", None)

## 4. Image-generation functions

In [5]:
generation_transform = transforms.Compose([
    transforms.Resize(
        (256, 256),
        interpolation=(
            transforms.InterpolationMode.BICUBIC
        ),
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5),
    ),
])


class ImagePathDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image = Image.open(
            self.image_paths[index]
        ).convert("RGB")

        return generation_transform(image)


def save_generated_images(
    model,
    input_paths,
    output_folder,
    description,
):
    output_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    for old_path in output_folder.glob("*.png"):
        old_path.unlink()

    dataset = ImagePathDataset(
        input_paths[:N_EVAL]
    )

    loader = DataLoader(
        dataset,
        batch_size=GENERATION_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    image_index = 0
    use_amp = device.type == "cuda"

    with torch.inference_mode():
        for batch in tqdm(loader, desc=description):
            batch = batch.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=use_amp,
            ):
                generated = model(batch)

            generated = generated.float().cpu()
            generated = (
                generated * 0.5 + 0.5
            ).clamp(0, 1)

            for image in generated:
                output_path = (
                    output_folder
                    / f"{image_index:05d}.png"
                )

                to_pil_image(image).save(output_path)
                image_index += 1

    return image_index

## 5. Official FID and MiFID functions

In [6]:
INCEPTION_TRANSFORM = transforms.Compose([
    transforms.Resize(299),
    transforms.CenterCrop(299),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225),
    ),
])


def get_inception_model():
    model = models.inception_v3(
        weights=(
            models.Inception_V3_Weights.IMAGENET1K_V1
        ),
        transform_input=False,
    )

    model.fc = nn.Identity()
    model = model.to(device)
    model.eval()

    return model


def load_inception_batch(paths):
    images = []

    for path in paths:
        image = Image.open(path).convert("RGB")
        images.append(INCEPTION_TRANSFORM(image))

    return torch.stack(images)


@torch.inference_mode()
def get_activations(
    model,
    image_paths,
    batch_size=32,
    description="Inception activations",
):
    features = []

    for start in tqdm(
        range(0, len(image_paths), batch_size),
        desc=description,
    ):
        batch_paths = image_paths[
            start:start + batch_size
        ]

        images = load_inception_batch(
            batch_paths
        ).to(device)

        batch_features = (
            model(images)
            .float()
            .cpu()
            .numpy()
        )

        features.append(batch_features)

    return np.concatenate(features, axis=0)


def frechet_distance(
    mu1,
    sigma1,
    mu2,
    sigma2,
    eps=1e-6,
):
    covmean = scipy.linalg.sqrtm(
        sigma1.dot(sigma2)
    )

    if not np.isfinite(covmean).all():
        offset = np.eye(sigma1.shape[0]) * eps

        covmean = scipy.linalg.sqrtm(
            (sigma1 + offset).dot(
                sigma2 + offset
            )
        )

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    difference = mu1 - mu2

    return float(
        difference.dot(difference)
        + np.trace(
            sigma1 + sigma2 - 2 * covmean
        )
    )


def calculate_fid_mifid_from_activations(
    real_activations,
    generated_activations,
):
    count = min(
        len(real_activations),
        len(generated_activations),
        N_EVAL,
    )

    real_activations = real_activations[:count]
    generated_activations = (
        generated_activations[:count]
    )

    real_mean = real_activations.mean(axis=0)
    real_covariance = np.cov(
        real_activations,
        rowvar=False,
    )

    generated_mean = generated_activations.mean(
        axis=0
    )
    generated_covariance = np.cov(
        generated_activations,
        rowvar=False,
    )

    fid = frechet_distance(
        real_mean,
        real_covariance,
        generated_mean,
        generated_covariance,
    )

    cosine_distances = [
        cosine(
            real_activations[index],
            generated_activations[index],
        )
        for index in range(count)
    ]

    mifid = float(np.mean(cosine_distances))

    return fid, mifid, count

## 6. Extract real-image features once

The same 300 real Monet images and 300 real photos are used for every checkpoint, so their Inception features only need to be calculated once.

In [7]:
inception_model = get_inception_model()

real_monet_activations = get_activations(
    inception_model,
    real_monet_paths,
    batch_size=METRIC_BATCH_SIZE,
    description="Real Monet activations",
)

real_photo_activations = get_activations(
    inception_model,
    real_photo_paths,
    batch_size=METRIC_BATCH_SIZE,
    description="Real photo activations",
)

print(
    "Real activations cached:",
    real_monet_activations.shape,
    real_photo_activations.shape,
)

Real photo activations: 100%|██████████| 10/10 [00:04<00:00,  2.40it/s]

Real activations cached: (300, 2048) (300, 2048)


## 7. Evaluate every checkpoint

In [8]:
all_results = []
failed_checkpoints = []


for checkpoint_path in checkpoint_paths:
    checkpoint_name = checkpoint_path.stem

    print("\n" + "=" * 70)
    print("Evaluating:", checkpoint_path.name)
    print("=" * 70)

    try:
        saved_epoch = load_generator_checkpoint(
            checkpoint_path
        )

        evaluation_root = (
            OUTPUT_ROOT / checkpoint_name
        )

        generated_photo_folder = (
            evaluation_root
            / "pred_A2B_monet_to_photo"
        )

        generated_monet_folder = (
            evaluation_root
            / "pred_B2A_photo_to_monet"
        )

        generated_photo_count = save_generated_images(
            model=G_BA,
            input_paths=real_monet_paths,
            output_folder=generated_photo_folder,
            description=(
                f"{checkpoint_name}: Monet -> photo"
            ),
        )

        generated_monet_count = save_generated_images(
            model=G_AB,
            input_paths=real_photo_paths,
            output_folder=generated_monet_folder,
            description=(
                f"{checkpoint_name}: photo -> Monet"
            ),
        )

        if (
            generated_photo_count != N_EVAL
            or generated_monet_count != N_EVAL
        ):
            raise RuntimeError(
                "The generated image count does not "
                f"match N_EVAL={N_EVAL}."
            )

        generated_photo_paths = list_images(
            generated_photo_folder
        )

        generated_monet_paths = list_images(
            generated_monet_folder
        )

        generated_monet_activations = get_activations(
            inception_model,
            generated_monet_paths,
            batch_size=METRIC_BATCH_SIZE,
            description=(
                f"{checkpoint_name}: generated Monet"
            ),
        )

        generated_photo_activations = get_activations(
            inception_model,
            generated_photo_paths,
            batch_size=METRIC_BATCH_SIZE,
            description=(
                f"{checkpoint_name}: generated photo"
            ),
        )

        photo_to_monet_fid, photo_to_monet_mifid, _ = (
            calculate_fid_mifid_from_activations(
                real_monet_activations,
                generated_monet_activations,
            )
        )

        monet_to_photo_fid, monet_to_photo_mifid, _ = (
            calculate_fid_mifid_from_activations(
                real_photo_activations,
                generated_photo_activations,
            )
        )

        average_fid = (
            photo_to_monet_fid
            + monet_to_photo_fid
        ) / 2.0

        average_mifid = (
            photo_to_monet_mifid
            + monet_to_photo_mifid
        ) / 2.0

        final_average_score = (
            average_fid + average_mifid
        ) / 2.0

        result = {
            "checkpoint": checkpoint_name,
            "saved_epoch": saved_epoch,
            "photo_to_monet_fid": (
                photo_to_monet_fid
            ),
            "photo_to_monet_mifid": (
                photo_to_monet_mifid
            ),
            "monet_to_photo_fid": (
                monet_to_photo_fid
            ),
            "monet_to_photo_mifid": (
                monet_to_photo_mifid
            ),
            "average_fid": average_fid,
            "average_mifid": average_mifid,
            "final_average_score": (
                final_average_score
            ),
        }

        all_results.append(result)

        directional_metrics = {
            "photo_to_monet": {
                "FID": float(photo_to_monet_fid),
                "MiFID": float(photo_to_monet_mifid),
                "evaluation_images": N_EVAL,
            },
            "monet_to_photo": {
                "FID": float(monet_to_photo_fid),
                "MiFID": float(monet_to_photo_mifid),
                "evaluation_images": N_EVAL,
            },
            "average": {
                "FID": float(average_fid),
                "MiFID": float(average_mifid),
                "final_average_score": float(
                    final_average_score
                ),
            },
        }

        with open(
            evaluation_root
            / "directional_metrics.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                directional_metrics,
                file,
                indent=4,
            )

        submission = pd.DataFrame([{
            "ID": 1,
            "FID": float(average_fid),
            "MiFID": float(average_mifid),
        }])

        submission.to_csv(
            evaluation_root / "submission.csv",
            index=False,
        )

        print(
            f"{checkpoint_name} | "
            f"Average FID: {average_fid:.4f} | "
            f"Average MiFID: {average_mifid:.4f} | "
            f"Final average: "
            f"{final_average_score:.4f}"
        )

    except Exception as error:
        failed_checkpoints.append({
            "checkpoint": checkpoint_name,
            "error": str(error),
        })

        print(
            f"FAILED: {checkpoint_name}: {error}"
        )

    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


if not all_results:
    raise RuntimeError(
        "Every checkpoint evaluation failed."
    )


Evaluating: generators_epoch_060.pt


generators_epoch_060: generated photo: 100%|██████████| 10/10 [00:06<00:00,  1.63it/s]


generators_epoch_060 | Average FID: 97.8528 | Average MiFID: 0.4069 | Final average: 49.1298

Evaluating: generators_epoch_065.pt


generators_epoch_065: generated photo: 100%|██████████| 10/10 [00:06<00:00,  1.56it/s]


generators_epoch_065 | Average FID: 97.5680 | Average MiFID: 0.4071 | Final average: 48.9876

Evaluating: generators_epoch_070.pt


...

generators_epoch_095 | Average FID: 96.0595 | Average MiFID: 0.4065 | Final average: 48.2330

Evaluating: generators_epoch_100.pt


generators_epoch_100: generated photo: 100%|██████████| 10/10 [00:05<00:00,  1.83it/s]


generators_epoch_100 | Average FID: 96.2489 | Average MiFID: 0.4067 | Final average: 48.3278

Evaluating: best_cycle_checkpoint.pt


best_cycle_checkpoint: generated photo: 100%|██████████| 10/10 [00:05<00:00,  1.76it/s]


best_cycle_checkpoint | Average FID: 96.8440 | Average MiFID: 0.4076 | Final average: 48.6258


## 8. Final checkpoint comparison

In [9]:
results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by="final_average_score",
    ascending=True,
).reset_index(drop=True)

results_df.insert(
    0,
    "rank",
    range(1, len(results_df) + 1),
)

results_csv_path = (
    OUTPUT_ROOT / "all_checkpoint_results.csv"
)

results_json_path = (
    OUTPUT_ROOT / "all_checkpoint_results.json"
)

results_df.to_csv(
    results_csv_path,
    index=False,
)

with open(
    results_json_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        all_results,
        file,
        indent=4,
    )

display_columns = [
    "rank",
    "checkpoint",
    "saved_epoch",
    "photo_to_monet_fid",
    "monet_to_photo_fid",
    "average_fid",
    "average_mifid",
    "final_average_score",
]

print("Final results, sorted by final average score:")

display(
    results_df[display_columns].round(4)
)

best_result = results_df.iloc[0]

print("Best checkpoint:", best_result["checkpoint"])
print(
    "Best average FID:",
    f"{best_result['average_fid']:.4f}",
)
print(
    "Best average MiFID:",
    f"{best_result['average_mifid']:.4f}",
)
print(
    "Best final average score:",
    f"{best_result['final_average_score']:.4f}",
)

print("Saved table:", results_csv_path)
print("Saved JSON:", results_json_path)

if failed_checkpoints:
    print("Failed checkpoints:")
    display(pd.DataFrame(failed_checkpoints))

Final results, sorted by final average score:


,rank,checkpoint,saved_epoch,photo_to_monet_fid,monet_to_photo_fid,average_fid,average_mifid,final_average_score
0,1,generators_epoch_085,85,93.5835,98.4486,96.0160,0.4067,48.2114
1,2,generators_epoch_095,95,93.5717,98.5474,96.0595,0.4065,48.2330
2,3,generators_epoch_080,80,93.6427,98.5613,96.1020,0.4068,48.2544
3,4,generators_epoch_100,100,93.4211,99.0767,96.2489,0.4067,48.3278
4,5,generators_epoch_075,75,94.0380,98.9458,96.4919,0.4068,48.4494
5,6,generators_epoch_090,90,93.8284,99.1969,96.5127,0.4067,48.4597
6,7,best_cycle_checkpoint,66,93.6688,100.0193,96.8440,0.4076,48.6258
7,8,generators_epoch_070,70,93.8414,100.0153,96.9284,0.4065,48.6674
8,9,generators_epoch_065,65,94.6862,100.4498,97.5680,0.4071,48.9876
9,10,generators_epoch_060,60,95.1494,100.5562,97.8528,0.4069,49.1298


Best checkpoint: generators_epoch_085
Best average FID: 96.0160
Best average MiFID: 0.4067
Best final average score: 48.2114
Saved table: /app/task3_gan_diffaug_ema/batch_official_evaluation_300/all_checkpoint_results.csv
Saved JSON: /app/task3_gan_diffaug_ema/batch_official_evaluation_300/all_checkpoint_results.json
